In [1]:
!pip -q install openpyxl

import os
import json
import logging
from pathlib import Path

BASE_DIR = Path("/content/agentic-spreadsheet-processor")

for folder in ["src", "configs", "data", "reports"]:
    (BASE_DIR / folder).mkdir(parents=True, exist_ok=True)

(BASE_DIR / "src" / "__init__.py").write_text("", encoding="utf-8")

os.chdir(BASE_DIR)
print("Project directory:", BASE_DIR)
print("Folders created successfully.")

Project directory: /content/agentic-spreadsheet-processor
Folders created successfully.


In [2]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "SalesData"

rows = [
    ["Product Name", "Quantity", "Unit Price"],
    ["Laptop", 2, 1200.50],
    ["Mouse", 5, 25.00],
    ["Keyboard", 3, 75.99],
    ["Monitor", 1, 300.00],
    ["Projector", 1, 850.00],
    ["Webcam", 10, 45.00],
    ["Headset", 4, 99.99],
    ["SSD", 2, 180.00],
    ["Router", 1, 70.00],
]

for row in rows:
    ws.append(row)

for cell in ws[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill("solid", fgColor="1F4E78")
    cell.alignment = Alignment(horizontal="center")

for row in range(2, ws.max_row + 1):
    ws.cell(row, 3).number_format = "#,##0.00"

for col in range(1, ws.max_column + 1):
    ws.column_dimensions[get_column_letter(col)].width = 20

input_path = BASE_DIR / "data" / "input_data.xlsx"
wb.save(input_path)

print("Created:", input_path)
ws

Created: /content/agentic-spreadsheet-processor/data/input_data.xlsx


<Worksheet "SalesData">

In [3]:
config = {
    "input_file": "data/input_data.xlsx",
    "output_file": "data/output_report.xlsx",
    "tasks": [
        {
            "type": "copy_data",
            "source_sheet": "SalesData",
            "dest_sheet": "Raw Data"
        },
        {
            "type": "calculate_column",
            "sheet": "Raw Data",
            "target_column": "D",
            "header": "Total Revenue",
            "start_row": 2,
            "formula": "=B{row}*C{row}",
            "number_format": "#,##0.00"
        },
        {
            "type": "conditional_format",
            "sheet": "Raw Data",
            "range": "D2:D100",
            "rule_type": "color_scale",
            "min_color": "F8696B",
            "mid_color": "FFEB84",
            "max_color": "63BE7B"
        },
        {
            "type": "calculate_column",
            "sheet": "Raw Data",
            "target_column": "E",
            "header": "Status",
            "start_row": 2,
            "formula": '=IF(D{row}>1000,"High Value","Standard")'
        },
        {
            "type": "conditional_format",
            "sheet": "Raw Data",
            "range": "E2:E100",
            "rule_type": "expression",
            "formula": '$E2="High Value"',
            "fill_color": "92D050",
            "font_color": "000000"
        },
        {
            "type": "summary",
            "sheet": "Raw Data",
            "summary_sheet": "Summary"
        },
        {
            "type": "chart",
            "sheet": "Summary",
            "chart_type": "bar",
            "data_range": "A1:B10",
            "category_range": "A2:A10",
            "title": "Product-wise Revenue",
            "x_axis_title": "Product",
            "y_axis_title": "Revenue",
            "top_left_cell": "D2"
        }
    ]
}

config_path = BASE_DIR / "configs" / "example_spreadsheet_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4, ensure_ascii=False)

print("Created:", config_path)
print(json.dumps(config, indent=2, ensure_ascii=False))

Created: /content/agentic-spreadsheet-processor/configs/example_spreadsheet_config.json
{
  "input_file": "data/input_data.xlsx",
  "output_file": "data/output_report.xlsx",
  "tasks": [
    {
      "type": "copy_data",
      "source_sheet": "SalesData",
      "dest_sheet": "Raw Data"
    },
    {
      "type": "calculate_column",
      "sheet": "Raw Data",
      "target_column": "D",
      "header": "Total Revenue",
      "start_row": 2,
      "formula": "=B{row}*C{row}",
      "number_format": "#,##0.00"
    },
    {
      "type": "conditional_format",
      "sheet": "Raw Data",
      "range": "D2:D100",
      "rule_type": "color_scale",
      "min_color": "F8696B",
      "mid_color": "FFEB84",
      "max_color": "63BE7B"
    },
    {
      "type": "calculate_column",
      "sheet": "Raw Data",
      "target_column": "E",
      "header": "Status",
      "start_row": 2,
      "formula": "=IF(D{row}>1000,\"High Value\",\"Standard\")"
    },
    {
      "type": "conditional_format",
   

In [4]:
%%writefile src/utils.py
import logging
import json
from pathlib import Path

def setup_logging(name, level=logging.INFO, filename=None):
    logger = logging.getLogger(name)
    logger.setLevel(level)

    if not logger.handlers:
        formatter = logging.Formatter(
            "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
        )

        console = logging.StreamHandler()
        console.setFormatter(formatter)
        logger.addHandler(console)

        if filename:
            file_handler = logging.FileHandler(filename, encoding="utf-8")
            file_handler.setFormatter(formatter)
            logger.addHandler(file_handler)

    return logger

def save_json_report(data, filename, directory="reports"):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    filepath = directory / filename

    try:
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
        return True
    except OSError:
        return False

Writing src/utils.py


In [5]:
%%writefile src/config_parser.py
import json
from pathlib import Path
from .utils import setup_logging

logger = setup_logging(__name__)

class ConfigParser:
    def __init__(self, config_path):
        self.config_path = Path(config_path)
        self.config = {}

    def load_config(self):
        if not self.config_path.exists():
            raise FileNotFoundError(
                f"Configuration not found: {self.config_path}"
            )

        try:
            with open(self.config_path, "r", encoding="utf-8") as f:
                self.config = json.load(f)
        except json.JSONDecodeError as e:
            raise ValueError(f"Invalid JSON configuration: {e}") from e

        self._validate_config()
        logger.info("Configuration loaded and validated.")
        return self.config

    def _validate_config(self):
        for field in ("input_file", "output_file", "tasks"):
            if field not in self.config:
                raise ValueError(f"Missing configuration field: {field}")

        if not isinstance(self.config["tasks"], list):
            raise ValueError("'tasks' must be a list.")

        for index, task in enumerate(self.config["tasks"], start=1):
            if not isinstance(task, dict) or "type" not in task:
                raise ValueError(f"Task {index} must be an object with 'type'.")

            if task["type"] == "chart":
                if not all(k in task for k in ("sheet", "chart_type", "data_range")):
                    raise ValueError(f"Chart task {index} is missing required fields.")

            if task["type"] == "conditional_format":
                if not all(k in task for k in ("sheet", "range", "rule_type")):
                    raise ValueError(f"Conditional format task {index} is incomplete.")

Writing src/config_parser.py


In [6]:
%%writefile src/spreadsheet_agent.py
import json
import logging
from datetime import datetime
from pathlib import Path

import openpyxl
from openpyxl.styles import Font, PatternFill
from openpyxl.formatting.rule import ColorScaleRule, FormulaRule
from openpyxl.chart import BarChart, LineChart, PieChart, Reference
from openpyxl.utils.cell import range_boundaries

from .utils import setup_logging

logger = setup_logging(__name__)

class SpreadsheetAgent:
    def __init__(self, config):
        self.config = config
        self.workbook = None
        self.audit_log = []

    def log_audit(self, status, task_type, message):
        self.audit_log.append({
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "status": status,
            "task_type": task_type,
            "message": message
        })

        log_method = logger.error if status == "ERROR" else logger.info
        log_method("%s | %s | %s", status, task_type, message)

    def get_sheet(self, name):
        if name not in self.workbook.sheetnames:
            return self.workbook.create_sheet(name)
        return self.workbook[name]

    def copy_data(self, task):
        source_name = task["source_sheet"]
        destination_name = task["dest_sheet"]

        if source_name not in self.workbook.sheetnames:
            raise ValueError(f"Source sheet not found: {source_name}")

        source = self.workbook[source_name]
        destination = self.get_sheet(destination_name)

        for row in source.iter_rows():
            for cell in row:
                destination[cell.coordinate] = cell.value

        self.log_audit(
            "SUCCESS", "copy_data",
            f"Copied {source_name} to {destination_name}"
        )

    def calculate_column(self, task):
        sheet = self.get_sheet(task["sheet"])
        column = task["target_column"]
        start_row = int(task.get("start_row", 2))
        formula = task["formula"]

        header = sheet[f"{column}1"]
        header.value = task.get("header", "Calculated Column")
        header.font = Font(bold=True, color="FFFFFF")
        header.fill = PatternFill("solid", fgColor="1F4E78")

        for row in range(start_row, sheet.max_row + 1):
            cell = sheet[f"{column}{row}"]
            cell.value = formula.replace("{row}", str(row))

            if task.get("number_format"):
                cell.number_format = task["number_format"]

        self.log_audit(
            "SUCCESS", "calculate_column",
            f"Applied formula to column {column} in {task['sheet']}"
        )

    def conditional_format(self, task):
        sheet = self.get_sheet(task["sheet"])
        cell_range = task["range"]
        rule_type = task["rule_type"]

        if rule_type == "color_scale":
            rule = ColorScaleRule(
                start_type="min",
                start_color=task.get("min_color", "F8696B"),
                mid_type="percentile",
                mid_value=50,
                mid_color=task.get("mid_color", "FFEB84"),
                end_type="max",
                end_color=task.get("max_color", "63BE7B")
            )
            sheet.conditional_formatting.add(cell_range, rule)

        elif rule_type == "expression":
            formula = task.get("formula")
            if not formula:
                raise ValueError("Expression rule requires a formula.")

            fill = PatternFill(
                "solid",
                fgColor=task.get("fill_color", "FFC7CE")
            )
            font = Font(color=task.get("font_color", "9C0006"))
            sheet.conditional_formatting.add(
                cell_range,
                FormulaRule(formula=[formula], fill=fill, font=font)
            )

        else:
            raise ValueError(f"Unsupported rule type: {rule_type}")

        self.log_audit(
            "SUCCESS", "conditional_format",
            f"Applied {rule_type} to {task['sheet']}!{cell_range}"
        )

    def create_summary(self, task):
        source = self.get_sheet(task["sheet"])
        summary = self.get_sheet(task.get("summary_sheet", "Summary"))

        # Remove old summary contents if rerun
        if summary.max_row:
            summary.delete_rows(1, summary.max_row)

        summary.append(["Product Name", "Total Revenue"])

        for row in range(2, source.max_row + 1):
            product = source.cell(row, 1).value
            revenue_cell = f"D{row}"

            if product is not None:
                summary.append([product, f"='Raw Data'!{revenue_cell}"])

        summary["D1"] = "Sales Summary"
        summary["D1"].font = Font(bold=True, size=14)

        for cell in summary[1][:2]:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = PatternFill("solid", fgColor="1F4E78")

        summary.column_dimensions["A"].width = 24
        summary.column_dimensions["B"].width = 20

        for row in range(2, summary.max_row + 1):
            summary.cell(row, 2).number_format = "#,##0.00"

        self.log_audit(
            "SUCCESS", "summary",
            f"Created summary sheet with {summary.max_row - 1} products"
        )

    def create_chart(self, task):
        sheet = self.get_sheet(task["sheet"])
        chart_type = task["chart_type"].lower()

        chart_classes = {
            "bar": BarChart,
            "line": LineChart,
            "pie": PieChart
        }

        if chart_type not in chart_classes:
            raise ValueError(f"Unsupported chart type: {chart_type}")

        chart = chart_classes[chart_type]()
        chart.title = task.get("title", "Sales Chart")

        if hasattr(chart, "x_axis"):
            chart.x_axis.title = task.get("x_axis_title", "")
        if hasattr(chart, "y_axis"):
            chart.y_axis.title = task.get("y_axis_title", "")

        min_col, min_row, max_col, max_row = range_boundaries(
            task["data_range"]
        )
        data = Reference(
            sheet, min_col=min_col, min_row=min_row,
            max_col=max_col, max_row=max_row
        )
        chart.add_data(data, titles_from_data=True)

        category_range = task.get("category_range")
        if category_range:
            min_col, min_row, max_col, max_row = range_boundaries(
                category_range
            )
            categories = Reference(
                sheet, min_col=min_col, min_row=min_row,
                max_col=max_col, max_row=max_row
            )
            chart.set_categories(categories)

        sheet.add_chart(chart, task.get("top_left_cell", "D2"))

        self.log_audit(
            "SUCCESS", "chart",
            f"Created {chart_type} chart: {chart.title}"
        )

    def add_audit_sheet(self):
        name = "Audit Log"
        if name in self.workbook.sheetnames:
            del self.workbook[name]

        sheet = self.workbook.create_sheet(name, 0)
        headers = ["Timestamp", "Status", "Task Type", "Message"]
        sheet.append(headers)

        for cell in sheet[1]:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = PatternFill("solid", fgColor="333333")

        for entry in self.audit_log:
            sheet.append([
                entry["timestamp"],
                entry["status"],
                entry["task_type"],
                entry["message"]
            ])

        for column_cells in sheet.columns:
            max_length = max(
                (len(str(cell.value or "")) for cell in column_cells),
                default=10
            )
            sheet.column_dimensions[
                column_cells[0].column_letter
            ].width = min(max_length + 2, 60)

    def run(self):
        input_file = Path(self.config["input_file"])
        output_file = Path(self.config["output_file"])

        try:
            if not input_file.exists():
                raise FileNotFoundError(f"Input workbook not found: {input_file}")

            self.workbook = openpyxl.load_workbook(input_file)
            self.log_audit("SUCCESS", "load_workbook", f"Loaded {input_file}")

            handlers = {
                "copy_data": self.copy_data,
                "calculate_column": self.calculate_column,
                "conditional_format": self.conditional_format,
                "summary": self.create_summary,
                "chart": self.create_chart
            }

            for index, task in enumerate(self.config["tasks"], start=1):
                task_type = task["type"]

                try:
                    if task_type not in handlers:
                        raise ValueError(f"Unsupported task type: {task_type}")

                    handlers[task_type](task)

                except Exception as error:
                    self.log_audit(
                        "ERROR", task_type,
                        f"Task {index} failed: {error}"
                    )

            self.log_audit(
                "INFO", "run_summary",
                f"Finished processing {len(self.config['tasks'])} tasks"
            )

            self.add_audit_sheet()
            output_file.parent.mkdir(parents=True, exist_ok=True)
            self.workbook.save(output_file)

            logger.info("Output saved to %s", output_file)
            return str(output_file)

        except Exception as error:
            self.log_audit("ERROR", "run", str(error))
            raise

Writing src/spreadsheet_agent.py


In [7]:
%%writefile main.py
from pathlib import Path

from src.config_parser import ConfigParser
from src.spreadsheet_agent import SpreadsheetAgent
from src.utils import setup_logging

BASE_DIR = Path(__file__).resolve().parent
logger = setup_logging("main")

def main():
    config_path = BASE_DIR / "configs" / "example_spreadsheet_config.json"

    try:
        config = ConfigParser(config_path).load_config()

        # Resolve file paths relative to the project directory
        config["input_file"] = str(BASE_DIR / config["input_file"])
        config["output_file"] = str(BASE_DIR / config["output_file"])

        agent = SpreadsheetAgent(config)
        output_path = agent.run()

        print("\nProcessing completed.")
        print("Output file:", output_path)
        print("Workbook sheets:", agent.workbook.sheetnames)
        print("Audit entries:", len(agent.audit_log))

    except (FileNotFoundError, ValueError) as error:
        logger.error("Cannot start processor: %s", error)
        raise
    except Exception as error:
        logger.exception("Unexpected processing error: %s", error)
        raise

if __name__ == "__main__":
    main()

Writing main.py


In [8]:
%cd /content/agentic-spreadsheet-processor
!python main.py

/content
2026-10-11 03:36:48,982 - src.config_parser - INFO - Configuration loaded and validated.
2026-10-11 03:36:48,998 - src.spreadsheet_agent - INFO - SUCCESS | load_workbook | Loaded /content/agentic-spreadsheet-processor/data/input_data.xlsx
2026-10-11 03:36:48,999 - src.spreadsheet_agent - INFO - SUCCESS | copy_data | Copied SalesData to Raw Data
2026-10-11 03:36:48,999 - src.spreadsheet_agent - INFO - SUCCESS | calculate_column | Applied formula to column D in Raw Data
2026-10-11 03:36:49,000 - src.spreadsheet_agent - INFO - SUCCESS | conditional_format | Applied color_scale to Raw Data!D2:D100
2026-10-11 03:36:49,000 - src.spreadsheet_agent - INFO - SUCCESS | calculate_column | Applied formula to column E in Raw Data
2026-10-11 03:36:49,001 - src.spreadsheet_agent - INFO - SUCCESS | conditional_format | Applied expression to Raw Data!E2:E100
2026-10-11 03:36:49,002 - src.spreadsheet_agent - INFO - SUCCESS | summary | Created summary sheet with 9 products
2026-10-11 03:36:49,00

In [9]:
import openpyxl
from pathlib import Path

output_path = BASE_DIR / "data" / "output_report.xlsx"

result_wb = openpyxl.load_workbook(output_path, data_only=False)

print("Sheets:", result_wb.sheetnames)

print("\n--- Raw Data ---")
raw_ws = result_wb["Raw Data"]

for row in raw_ws.iter_rows(values_only=True):
    print(row)

print("\n--- Summary ---")
summary_ws = result_wb["Summary"]

for row in summary_ws.iter_rows(min_row=1, max_row=5, values_only=True):
    print(row)

print("\n--- Audit Log ---")
audit_ws = result_wb["Audit Log"]

for row in audit_ws.iter_rows(min_row=1, max_row=8, values_only=True):
    print(row)

Sheets: ['Audit Log', 'SalesData', 'Raw Data', 'Summary']

--- Raw Data ---
('Product Name', 'Quantity', 'Unit Price', 'Total Revenue', 'Status')
('Laptop', 2, 1200.5, '=B2*C2', '=IF(D2>1000,"High Value","Standard")')
('Mouse', 5, 25, '=B3*C3', '=IF(D3>1000,"High Value","Standard")')
('Keyboard', 3, 75.99, '=B4*C4', '=IF(D4>1000,"High Value","Standard")')
('Monitor', 1, 300, '=B5*C5', '=IF(D5>1000,"High Value","Standard")')
('Projector', 1, 850, '=B6*C6', '=IF(D6>1000,"High Value","Standard")')
('Webcam', 10, 45, '=B7*C7', '=IF(D7>1000,"High Value","Standard")')
('Headset', 4, 99.99, '=B8*C8', '=IF(D8>1000,"High Value","Standard")')
('SSD', 2, 180, '=B9*C9', '=IF(D9>1000,"High Value","Standard")')
('Router', 1, 70, '=B10*C10', '=IF(D10>1000,"High Value","Standard")')

--- Summary ---
('Product Name', 'Total Revenue', None, 'Sales Summary')
('Laptop', "='Raw Data'!D2", None, None)
('Mouse', "='Raw Data'!D3", None, None)
('Keyboard', "='Raw Data'!D4", None, None)
('Monitor', "='Raw Data'

In [10]:
result_wb = openpyxl.load_workbook(output_path, data_only=False)

print("Charts in Summary:", len(result_wb["Summary"]._charts))
print("Charts in Raw Data:", len(result_wb["Raw Data"]._charts))
print("Conditional formatting rules:",
      len(result_wb["Raw Data"].conditional_formatting))

audit_ws = result_wb["Audit Log"]

success_count = 0
error_count = 0

for row in audit_ws.iter_rows(min_row=2, values_only=True):
    status = row[1]
    if status == "SUCCESS":
        success_count += 1
    elif status == "ERROR":
        error_count += 1

print("Successful audit entries:", success_count)
print("Error audit entries:", error_count)
print("Output file size:", round(output_path.stat().st_size / 1024, 2), "KB")

Charts in Summary: 1
Charts in Raw Data: 0
Conditional formatting rules: 2
Successful audit entries: 8
Error audit entries: 0
Output file size: 10.19 KB


In [11]:
from google.colab import files

files.download(str(output_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>